In [1]:
# =====================================
# IMPORTS
# =====================================

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

In [2]:
# =====================================
# LEITURA DAS BASES
# =====================================

path_19_24 = r"C:\Users\x15501492\Documents\02 - Publicações\11 - Publicação SESP - Site\2026\01 - Janeiro\Excel\19_24_agrupado_crimes_violentos.xlsx"
path_25_26 = r"C:\Users\x15501492\Documents\02 - Publicações\11 - Publicação SESP - Site\2026\01 - Janeiro\Excel\25_26_agrupado_crimes_violentos.xlsx"
path_homicidio = r"C:\Users\x15501492\Documents\02 - Publicações\11 - Publicação SESP - Site\2026\01 - Janeiro\Excel\agrupado_vitimas_homicidio_consumado.xlsx"
path_pop = r"C:\Users\x15501492\OneDrive - CAMG\DIS -  Henrique\Dados abertos\Auxiliares\populacao.xlsx"

df_19_24 = pd.read_excel(path_19_24)
df_25_26 = pd.read_excel(path_25_26)
df_pop = pd.read_excel(path_pop)
df_homicidio = pd.read_excel(path_homicidio)
df_cv = pd.concat([df_19_24, df_25_26], ignore_index=True)

df_cv.head()
df_homicidio.head()

,Registros,Natureza,Município,Cod. IBGE,Mês,Ano,RISP,RMBH
0,0,Homicídio Consumado (Vitimas),ABADIA DOS DOURADOS,310010,1,2012,RISP 10 - PATOS DE MINAS,NÃO
1,0,Homicídio Consumado (Vitimas),ABAETE,310020,1,2012,RISP 7 - DIVINÓPOLIS,NÃO
2,0,Homicídio Consumado (Vitimas),ABRE-CAMPO,310030,1,2012,RISP 12 - IPATINGA,NÃO
3,0,Homicídio Consumado (Vitimas),ACAIACA,310040,1,2012,RISP 12 - IPATINGA,NÃO
4,0,Homicídio Consumado (Vitimas),ACUCENA,310050,1,2012,RISP 12 - IPATINGA,NÃO


In [3]:
# =====================================
# PADRONIZA COLUNA ANO
# =====================================

df_pop = df_pop.rename(columns={
    "ano": "Ano",
    "pop mg": "População"
})

df_cv = df_cv.rename(columns={"Ano Fato": "Ano"})
df_cv.head()

,Registros,Natureza,Município,Cód. IBGE,Mês,Ano,RISP,RMBH
0,0,ESTUPRO CONSUMADO,ABADIA DOS DOURADOS,310010,1,2019,RISP 10 - PATOS DE MINAS,NÃO
1,0,ESTUPRO CONSUMADO,ABAETE,310020,1,2019,RISP 7 - DIVINÓPOLIS,NÃO
2,0,ESTUPRO CONSUMADO,ABRE-CAMPO,310030,1,2019,RISP 12 - IPATINGA,NÃO
3,0,ESTUPRO CONSUMADO,ACAIACA,310040,1,2019,RISP 12 - IPATINGA,NÃO
4,0,ESTUPRO CONSUMADO,ACUCENA,310050,1,2019,RISP 12 - IPATINGA,NÃO


In [ ]:
df_pop.head()

,Ano,População
0,2019,21168791
1,2020,21292666
2,2021,21411923
3,2022,20732660
4,2023,20732660


In [5]:
# =========================================
# REGISTRO E TAXA DE CRIMES VIOLENTOS 20/25
# =========================================

df_cv_20_25 = df_cv[
    (df_cv["Ano"] >= 2020) &
    (df_cv["Ano"] <= 2025)
].copy()

cv_ano = (
    df_cv_20_25
    .groupby("Ano")["Registros"]
    .sum()
    .reset_index(name="Crimes Violentos")
)

mun_ano = (
    df_cv_20_25
    .groupby(["Ano", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_crime = (
    mun_ano
    .query("Registros == 0")
    .groupby("Ano")["Município"]
    .count()
    .reset_index(name="Municípios sem Crimes Violentos")
)

tb_cv_20_25 = cv_ano.merge(df_pop, on="Ano", how="left")

tb_cv_20_25["Taxa por 100 mil"] = (
    tb_cv_20_25["Crimes Violentos"] / tb_cv_20_25["População"] * 100000
)

tb_cv_20_25 = tb_cv_20_25.merge(
    mun_sem_crime,
    on="Ano",
    how="left"
)

tb_cv_20_25 = tb_cv_20_25[
    [
        "Ano",
        "Crimes Violentos",
        "Taxa por 100 mil",
        "Municípios sem Crimes Violentos"
    ]
]

tb_cv_20_25.style.format({
    "Crimes Violentos": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Ano,Crimes Violentos,Taxa por 100 mil,Municípios sem Crimes Violentos
0,2020,"47,499",223.08,43
1,2021,"37,959",177.28,45
2,2022,"36,940",178.17,62
3,2023,"32,648",157.47,65
4,2024,"32,957",154.56,44
5,2025,"26,123",122.51,65


In [6]:
# =========================================
# REGISTRO E TAXA DE CRIMES VIOLENTOS 2026
# =========================================

df_cv_26 = df_cv[
    (df_cv["Ano"] >= 2026)
].copy()

cv_2026 = (
    df_cv_26
    .groupby("Mês")["Registros"]
    .sum()
    .reset_index(name="Crimes Violentos")
)

mun_ano = (
    df_cv_26
    .groupby(["Mês", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_crime = (
    mun_ano
    .query("Registros == 0")
    .groupby("Mês")["Município"]
    .count()
    .reset_index(name="Municípios sem Crimes Violentos")
)

pop_2026 = df_pop.loc[df_pop["Ano"] == 2026, "População"].values[0]

cv_2026["População"] = pop_2026

cv_2026["Taxa por 100 mil"] = (
   cv_2026["Crimes Violentos"] / pop_2026 * 100000
)

tb_cv_26 = cv_2026.merge(
    mun_sem_crime,
    on="Mês",
    how="left"
)

tb_cv_26 = tb_cv_26[
    [
        "Mês",
        "Crimes Violentos",
        "Taxa por 100 mil",
        "Municípios sem Crimes Violentos"
    ]
]

tb_cv_26.style.format({
    "Crimes Violentos": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Mês,Crimes Violentos,Taxa por 100 mil,Municípios sem Crimes Violentos
0,1,"1,850",8.68,515


In [7]:
# =========================================
# REGISTRO E TAXA DE CVCP 20/25
# =========================================

naturezas_cvcp = [
    "EXTORSAO CONSUMADO",
    "EXTORSAO MEDIANTE SEQUESTRO CONSUMADO",
    "EXTORSAO TENTADO",
    "ROUBO TENTADO",
    "ROUBO CONSUMADO"
]

df_cvcp_20_25 = df_cv[
    (df_cv["Ano"] >= 2020) &
    (df_cv["Ano"] <= 2025) &
    (df_cv["Natureza"].isin(naturezas_cvcp))
].copy()

crimes_ano = (
    df_cvcp_20_25
    .groupby("Ano")["Registros"]
    .sum()
    .reset_index(name="Crimes Violentos Contra o Patrimônio")
)

mun_ano = (
    df_cvcp_20_25
    .groupby(["Ano", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_crime = (
    mun_ano
    .query("Registros == 0")
    .groupby("Ano")["Município"]
    .count()
    .reset_index(name="Municípios sem Crimes Violentos Contra o Patrimônio")
)

tb_cvcp_20_25 = crimes_ano.merge(df_pop, on="Ano", how="left")

tb_cvcp_20_25["Taxa por 100 mil"] = (
    tb_cvcp_20_25["Crimes Violentos Contra o Patrimônio"] / tb_cvcp_20_25["População"] * 100000
)

tb_cvcp_20_25 = tb_cvcp_20_25.merge(
    mun_sem_crime,
    on="Ano",
    how="left"
)

tb_cvcp_20_25 = tb_cvcp_20_25[
    [
        "Ano",
        "Crimes Violentos Contra o Patrimônio",
        "Taxa por 100 mil",
        "Municípios sem Crimes Violentos Contra o Patrimônio"
    ]
]

tb_cvcp_20_25.style.format({
    "Crimes Violentos Contra o Patrimônio": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Ano,Crimes Violentos Contra o Patrimônio,Taxa por 100 mil,Municípios sem Crimes Violentos Contra o Patrimônio
0,2020,"37,557",176.38,146
1,2021,"28,134",131.39,169
2,2022,"27,038",130.41,195
3,2023,"22,171",106.94,213
4,2024,"21,158",99.23,242
5,2025,"15,506",72.72,271


In [8]:
# =========================================
# REGISTRO E TAXA DE CVCP 2026
# =========================================

naturezas_cvcp = [
    "EXTORSAO CONSUMADO",
    "EXTORSAO MEDIANTE SEQUESTRO CONSUMADO",
    "EXTORSAO TENTADO",
    "ROUBO TENTADO",
    "ROUBO CONSUMADO"
]

df_cvcp_26 = df_cv[
    (df_cv["Ano"] >= 2026) &
    (df_cv["Natureza"].isin(naturezas_cvcp))
].copy()

cvcp_2026 = (
    df_cvcp_26
    .groupby("Mês")["Registros"]
    .sum()
    .reset_index(name="Crimes Violentos Contra o Patrimônio")
)

mun_ano = (
    df_cvcp_26
    .groupby(["Mês", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_crime = (
    mun_ano
    .query("Registros == 0")
    .groupby("Mês")["Município"]
    .count()
    .reset_index(name="Municípios sem Crimes Violentos Contra o Patrimônio")
)

pop_2026 = df_pop.loc[df_pop["Ano"] == 2026, "População"].values[0]

cvcp_2026["População"] = pop_2026

cvcp_2026["Taxa por 100 mil"] = (
   cvcp_2026["Crimes Violentos Contra o Patrimônio"] / pop_2026 * 100000
)

tb_cvcp_26 = cvcp_2026.merge(
    mun_sem_crime,
    on="Mês",
    how="left"
)

tb_cvcp_26 = tb_cvcp_26[
    [
        "Mês",
        "Crimes Violentos Contra o Patrimônio",
        "Taxa por 100 mil",
        "Municípios sem Crimes Violentos Contra o Patrimônio"
    ]
]

tb_cvcp_26.style.format({
    "Crimes Violentos Contra o Patrimônio": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Mês,Crimes Violentos Contra o Patrimônio,Taxa por 100 mil,Municípios sem Crimes Violentos Contra o Patrimônio
0,1,"1,116",5.23,663


In [9]:
# =========================================
# REGISTRO E TAXA DE HC 23/25
# =========================================

df_hc_23_25 = df_homicidio[
    (df_homicidio["Ano"] >= 2023) &
    (df_homicidio["Ano"] <= 2025)
].copy()

hc_ano = (
    df_hc_23_25
    .groupby("Ano")["Registros"]
    .sum()
    .reset_index(name="Vítimas de Homicídio Consumado")
)

mun_ano = (
    df_hc_23_25
    .groupby(["Ano", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_hc = (
    mun_ano
    .query("Registros == 0")
    .groupby("Ano")["Município"]
    .count()
    .reset_index(name="Municípios sem Vítimas de Homicídio Consumado")
)

tb_hc_23_25 = hc_ano.merge(df_pop, on="Ano", how="left")

tb_hc_23_25["Taxa por 100 mil"] = (
    tb_hc_23_25["Vítimas de Homicídio Consumado"] / tb_hc_23_25["População"] * 100000
)

tb_hc_23_25 = tb_hc_23_25.merge(
    mun_sem_hc,
    on="Ano",
    how="left"
)

tb_hc_23_25 = tb_hc_23_25[
    [
        "Ano",
        "Vítimas de Homicídio Consumado",
        "Taxa por 100 mil",
        "Municípios sem Vítimas de Homicídio Consumado"
    ]
]

tb_hc_23_25.style.format({
    "Vítimas de Homicídio Consumado": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Ano,Vítimas de Homicídio Consumado,Taxa por 100 mil,Municípios sem Vítimas de Homicídio Consumado
0,2023,"2,607",12.57,382
1,2024,"2,778",13.03,394
2,2025,"2,401",11.26,425


In [11]:
# =========================================
# REGISTRO E TAXA DE HC 2026
# =========================================

df_hc_26 = df_homicidio[
    (df_homicidio["Ano"] >= 2026)
].copy()

hc_26 = (
    df_hc_26
    .groupby("Ano")["Registros"]
    .sum()
    .reset_index(name="Vítimas de Homicídio Consumado")
)

mun_26 = (
    df_hc_26
    .groupby(["Ano", "Município"])["Registros"]
    .sum()
    .reset_index()
)

mun_sem_hc = (
    mun_26
    .query("Registros == 0")
    .groupby("Ano")["Município"]
    .count()
    .reset_index(name="Municípios sem Vítimas de Homicídio Consumado")
)

pop_2026 = df_pop.loc[df_pop["Ano"] == 2026, "População"].values[0]

hc_26["População"] = pop_2026

hc_26["Taxa por 100 mil"] = (
   hc_26["Vítimas de Homicídio Consumado"] / pop_2026 * 100000
)


tb_hc_26 = hc_26.merge(df_pop, on="Ano", how="left")

tb_hc_26 = tb_hc_26.merge(
    mun_sem_hc,
    on="Ano",
    how="left"
)

tb_hc_26 = tb_hc_26[
    [
        "Ano",
        "Vítimas de Homicídio Consumado",
        "Taxa por 100 mil",
        "Municípios sem Vítimas de Homicídio Consumado"
    ]
]

tb_hc_26.style.format({
    "Vítimas de Homicídio Consumado": "{:,.0f}",
    "Taxa por 100 mil": "{:,.2f}"
})

,Ano,Vítimas de Homicídio Consumado,Taxa por 100 mil,Municípios sem Vítimas de Homicídio Consumado
0,2026,157,0.74,764


In [16]:
import math
from openpyxl import Workbook
from openpyxl.styles import Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

In [17]:
# =========================================
# MUNICÍPIOS SEM VÍTIMAS DE HC EM 2026
# =========================================

municipios_sem_hc_2026 = (
    df_homicidio
    .query("Ano == 2026")
    .groupby("Município")["Registros"]
    .sum()
    .reset_index()
    .query("Registros == 0")
    ["Município"]
    .sort_values()
)

municipios_sem_hc_2026

0         ABADIA DOS DOURADOS
1                      ABAETE
2                  ABRE-CAMPO
3                     ACAIACA
4                     ACUCENA
                ...          
848              VIRGINOPOLIS
849               VIRGOLANDIA
850    VISCONDE DO RIO BRANCO
851              VOLTA GRANDE
852            WENCESLAU BRAZ
Name: Município, Length: 764, dtype: str

In [18]:
lista = list(municipios_sem_hc_2026)

linhas_por_coluna = 100
num_colunas = math.ceil(len(lista) / linhas_por_coluna)

colunas = {}

for i in range(num_colunas):
    inicio = i * linhas_por_coluna
    fim = inicio + linhas_por_coluna
    colunas[f"Coluna {i+1}"] = lista[inicio:fim]

df_final = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in colunas.items()]))

df_final

,Coluna 1,Coluna 2,Coluna 3,Coluna 4,Coluna 5,Coluna 6,Coluna 7,Coluna 8
0,ABADIA DOS DOURADOS,CAMACHO,CORREGO DO BOM JESUS,GUIMARANIA,LIMEIRA DO OESTE,PASSABEM,SANTA EFIGENIA DE MINAS,SERICITA
1,ABAETE,CAMANDUCAIA,CORREGO FUNDO,GUIRICEMA,LONTRA,PASSOS,SANTA FE DE MINAS,SERITINGA
2,ABRE-CAMPO,CAMBUI,CORREGO NOVO,GURINHATA,LUISBURGO,PATIS,SANTA HELENA DE MINAS,SERRA AZUL DE MINAS
3,ACAIACA,CAMBUQUIRA,COUTO DE MAG.DE MINAS,HELIODORA,LUISLANDIA,PATROCINIO,SANTA JULIANA,SERRA DA SAUDADE
4,ACUCENA,CAMPANARIO,CRISOLITA,IAPU,LUMINARIAS,PATROCINIO DO MURIAE,SANTA MARGARIDA,SERRA DO SALITRE
...,...,...,...,...,...,...,...,...
95,CAETANOPOLIS,COROMANDEL,GUARANI,LEANDRO FERREIRA,PARAISOPOLIS,SANTA BARBARA DO MONTE VERDE,SENADOR JOSE BENTO,NaN
96,CAETE,CORONEL MURTA,GUARARA,LEME DO PRADO,PARAOPEBA,SANTA BARBARA DO TUGURIO,SENADOR MODESTINO GONCALVES,NaN
97,CAIANA,CORONEL PACHECO,GUARDA-MOR,LEOPOLDINA,PASSA QUATRO,SANTA CRUZ DE MINAS,SENHORA DE OLIVEIRA,NaN
98,CAJURI,CORONEL XAVIER CHAVES,GUAXUPE,LIBERDADE,PASSA TEMPO,SANTA CRUZ DE SALINAS,SENHORA DO PORTO,NaN


In [20]:
# Criar workbook
wb = Workbook()
ws = wb.active

# Inserir dados
for r in dataframe_to_rows(df_final, index=False, header=False):
    ws.append(r)

# Criar estilo de borda
thin = Side(style="thin")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# Aplicar borda em todas as células preenchidas
for row in ws.iter_rows(
    min_row=1,
    max_row=ws.max_row,
    min_col=1,
    max_col=ws.max_column
):
    for cell in row:
        if cell.value is not None:
            cell.border = border

# Ajustar largura automática simples
for col in ws.columns:
    ws.column_dimensions[col[0].column_letter].width = 22

# Salvar arquivo
wb.save(r"C:\Users\x15501492\Downloads\municipios_sem_homicidio_2026.xlsx")